# Plant Disease Classification Pipeline

This notebook builds a reproducible PlantVillage classifier for a Kaggle GPU session. It keeps the experiment loop readable: inspect the files, create one fixed split, compare three model families, run two focused feature experiments, tune the best transfer model, and evaluate the selected model once on the held-out test set.

## Plan

1. Set seeds and a single configuration.
2. Locate the Kaggle dataset and build a class-count table.
3. Merge the supplied folders and create a fixed 70/15/15 path split.
4. Build raw-image, augmentation, preprocessing, and evaluation helpers.
5. Train a scratch CNN plus MobileNetV2 and EfficientNetB0 transfer models.
6. Run a contrast ablation and a frozen-embedding softmax experiment.
7. Tune the best transfer family on a 20% training subset.
8. Evaluate the final model once on test data and save figures, tables, and the model.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

The first code cell fixes Python, NumPy, and TensorFlow randomness. Keeping the seed in one place makes the split and the experiment comparisons easier to reproduce.

In [ ]:
CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "stage1_epochs": 8,
    "stage2_epochs": 5,
    "ablation_stage1_epochs": 4,
    "ablation_stage2_epochs": 3,
    "tuning_subset_fraction": 0.20,
    "tuning_trials": 6,
    "tuning_epochs": 3,
    "scratch_learning_rate": 1e-3,
    "transfer_learning_rate": 1e-3,
    "fine_tune_learning_rate": 1e-5,
    "tuning_learning_rates": [1e-4, 3e-4, 1e-3],
    "dropout_values": [0.2, 0.4, 0.6],
    "unfrozen_layer_values": [20, 40, 80],
    "cache_datasets": True,
}

IMAGE_SIZE = CONFIG["image_size"]
BATCH_SIZE = CONFIG["batch_size"]
NUM_CLASSES = None
MODEL_NAMES = ["baseline_cnn", "mobilenetv2", "efficientnetb0"]
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("GPU detected; mixed precision enabled.")
else:
    print("GPU not detected; using float32.")

All tunable values live in this configuration cell. The epoch counts and tuning subset are intentionally modest so the complete comparison can fit in one Kaggle GPU session; the final numbers must come from your own run.

## Locate the Dataset

The notebook searches under `/kaggle/input` for a directory containing both `train/` and `val/`. This avoids depending on one particular Kaggle mount depth while still failing early if the expected dataset is missing.

In [ ]:
def find_dataset_root(search_root=Path("/kaggle/input")):
    if not search_root.exists():
        raise FileNotFoundError("/kaggle/input does not exist. Attach the PlantVillage dataset to this Kaggle notebook.")
    candidates = [path for path in search_root.rglob("*") if path.is_dir() and (path / "train").is_dir() and (path / "val").is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not find a folder containing train/ and val/ under /kaggle/input.")
    return sorted(candidates, key=lambda path: len(path.parts))[0]


DATASET_ROOT = find_dataset_root()
print(f"Dataset root: {DATASET_ROOT}")

In [ ]:
def collect_image_records(dataset_root):
    rows = []
    for source_name in ["train", "val"]:
        source_dir = dataset_root / source_name
        for class_dir in sorted(path for path in source_dir.iterdir() if path.is_dir()):
            for image_path in sorted(path for path in class_dir.rglob("*") if path.is_file()):
                if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                    rows.append({"path": str(image_path), "label": class_dir.name, "source": source_name})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("No supported image files were found in train/ and val/.")
    return frame


all_df = collect_image_records(DATASET_ROOT)
class_names = sorted(all_df["label"].unique())
class_to_index = {name: index for index, name in enumerate(class_names)}
all_df["label_id"] = all_df["label"].map(class_to_index).astype("int32")
NUM_CLASSES = len(class_names)
print(f"Images: {len(all_df):,}; classes: {NUM_CLASSES}")

## Minimal EDA

The required EDA is deliberately small: one class-count table and bar chart, followed by one 3x4 image grid. This is enough to verify labels and file readability without turning the notebook into a collection of unrelated plots.

In [ ]:
class_counts = all_df["label"].value_counts().sort_index()
display(pd.DataFrame({"class": class_counts.index, "image_count": class_counts.values}))
print(f"Number of classes: {NUM_CLASSES}")

plt.figure(figsize=(14, 5))
class_counts.plot(kind="bar")
plt.xlabel("Class")
plt.ylabel("Images")
plt.title("Images per class")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
def show_sample_grid(frame, rows=3, columns=4):
    sample = frame.sample(n=min(rows * columns, len(frame)), random_state=SEED)
    figure, axes = plt.subplots(rows, columns, figsize=(14, 9))
    axes = np.asarray(axes).reshape(rows, columns)
    for axis in axes.flat:
        axis.axis("off")
    for axis, (_, record) in zip(axes.flat, sample.iterrows()):
        image = tf.io.decode_image(tf.io.read_file(record["path"]), channels=3, expand_animations=False)
        axis.imshow(image.numpy())
        axis.set_title(record["label"], fontsize=8)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


show_sample_grid(all_df)

The count check provides the evidence used later for class weighting. The sample grid is only a visual sanity check that paths, labels, and image decoding agree.

## Fixed Path Split and Near-Duplicate Check

The original train/val folders are merged before splitting, so the held-out test set is created by this notebook rather than inherited from the source layout. Splitting file paths before augmentation prevents augmented copies from crossing split boundaries.

In [ ]:
def filename_group_key(path):
    stem = Path(path).stem.casefold()
    for marker in ["_", "-", "("]:
        if marker in stem:
            prefix, suffix = stem.rsplit(marker, 1)
            if suffix.rstrip(")").isdigit() and prefix:
                return prefix
    return stem


all_df["group_key"] = all_df["path"].map(filename_group_key)
group_sizes = all_df.groupby(["label", "group_key"]).size()
grouping_available = bool((group_sizes > 1).any())
if grouping_available:
    print("Filename grouping detected; groups will stay inside one split.")
else:
    print("No reliable repeated filename groups were detected. Near-duplicate grouping remains a limitation of this dataset.")


def split_one_label(label_frame):
    if grouping_available:
        units = [unit for _, unit in label_frame.groupby("group_key")]
    else:
        units = [unit for _, unit in label_frame.groupby("path")]
    rng = np.random.default_rng(SEED + int(label_frame["label_id"].iloc[0]))
    rng.shuffle(units)
    targets = np.array([0.70, 0.15, 0.15]) * len(label_frame)
    selections = [[], [], []]
    totals = np.zeros(3, dtype=int)
    for unit in units:
        choice = int(np.argmin(totals - targets))
        selections[choice].append(unit)
        totals[choice] += len(unit)
    return [pd.concat(selection, ignore_index=True) for selection in selections]


split_parts = [[], [], []]
for _, label_frame in all_df.groupby("label", sort=True):
    for index, part in enumerate(split_one_label(label_frame)):
        split_parts[index].append(part)
train_df, val_df, test_df = [pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True) for parts in split_parts]

path_sets = [set(train_df["path"]), set(val_df["path"]), set(test_df["path"])]
assert not (path_sets[0] & path_sets[1] or path_sets[0] & path_sets[2] or path_sets[1] & path_sets[2])
split_sizes = pd.DataFrame({"split": ["train", "val", "test"], "images": [len(train_df), len(val_df), len(test_df)]})
display(split_sizes)
print("No image path appears in more than one split.")

The split is stratified by class and uses groups when repeated filename stems provide a defensible grouping signal. If they do not, the notebook states that limitation rather than pretending that filenames identify individual leaves.

In [ ]:
imbalance_ratio = class_counts.max() / class_counts.min()
use_class_weights = imbalance_ratio >= 1.5
class_weight = None
if use_class_weights:
    train_counts = train_df["label_id"].value_counts().sort_index()
    class_weight = {int(index): float(len(train_df) / (NUM_CLASSES * count)) for index, count in train_counts.items()}
print(f"Class-count imbalance ratio: {imbalance_ratio:.2f}")
print(f"Class weights used: {use_class_weights}")

## Data and Model Helpers

The shared helpers keep the experiment cells short. Images remain raw until the model-specific preprocessing function is applied, which prevents EfficientNetB0 or MobileNetV2 preprocessing from being accidentally applied twice.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.10),
], name="train_augmentation")


def decode_resize(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32), tf.cast(label, tf.int32)


def preprocess_images(images, model_name, extra_contrast=False):
    if extra_contrast:
        images = tf.image.adjust_contrast(images, contrast_factor=1.15)
    if model_name == "mobilenetv2":
        return tf.keras.applications.mobilenet_v2.preprocess_input(images)
    if model_name == "efficientnetb0":
        return images
    return images / 255.0


def build_dataset(frame, model_name, training=False, extra_contrast=False, cache_name=None):
    paths = frame["path"].to_numpy()
    labels = frame["label_id"].to_numpy(dtype="int32")
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if CONFIG["cache_datasets"]:
        dataset = dataset.cache() if cache_name is None else dataset.cache(str(OUTPUT_DIR / cache_name))
    if training:
        dataset = dataset.map(lambda image, label: (augmentation(image, training=True), label), num_parallel_calls=AUTOTUNE)
    dataset = dataset.map(lambda image, label: (preprocess_images(image, model_name, extra_contrast), label), num_parallel_calls=AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


def make_callbacks(name):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(OUTPUT_DIR / f"{name}.keras", monitor="val_loss", save_best_only=True),
    ]

In [ ]:
def build_baseline():
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation=None)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(0.20)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return tf.keras.Model(inputs, outputs, name="baseline_cnn")


def build_transfer_model(model_name, dropout=0.30, unfrozen_layers=0):
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    if model_name == "mobilenetv2":
        backbone = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
    elif model_name == "efficientnetb0":
        backbone = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
    else:
        raise ValueError(f"Unknown transfer model: {model_name}")
    backbone.trainable = unfrozen_layers > 0
    if unfrozen_layers > 0:
        for layer in backbone.layers[:-unfrozen_layers]:
            layer.trainable = False
        for layer in backbone.layers[-unfrozen_layers:]:
            layer.trainable = True
    x = backbone(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return tf.keras.Model(inputs, outputs, name=model_name)


def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def fit_model(model, train_data, val_data, name, epochs, learning_rate):
    compile_model(model, learning_rate)
    return model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        class_weight=class_weight,
        callbacks=make_callbacks(name),
        verbose=1,
    )


def fit_transfer_two_stage(model_name, train_data, val_data, name, dropout=0.30, unfrozen_layers=0, stage1_epochs=None, stage2_epochs=None):
    model = build_transfer_model(model_name, dropout=dropout, unfrozen_layers=0)
    stage1 = fit_model(model, train_data, val_data, f"{name}_stage1", stage1_epochs or CONFIG["stage1_epochs"], CONFIG["transfer_learning_rate"])
    if stage2_epochs:
        model = build_transfer_model(model_name, dropout=dropout, unfrozen_layers=unfrozen_layers or 40)
        model.set_weights(tf.keras.models.load_model(OUTPUT_DIR / f"{name}_stage1.keras").get_weights())
        stage2 = fit_model(model, train_data, val_data, f"{name}_stage2", stage2_epochs, CONFIG["fine_tune_learning_rate"])
    else:
        stage2 = None
    return model, stage1, stage2

The data pipeline caches resized images, augments only the training split, and prefetches with TensorFlow autotuning. The model builders cover a readable scratch CNN and two Keras application backbones, with a lower learning rate reserved for fine-tuning.

## Build the Three Splits

Each model sees the same path split and label mapping. This keeps the comparison focused on architecture and training strategy rather than different data exposure.

In [ ]:
train_data_by_model = {name: build_dataset(train_df, name, training=True, cache_name=f"train_{name}.cache") for name in MODEL_NAMES}
val_data_by_model = {name: build_dataset(val_df, name, cache_name=f"val_{name}.cache") for name in MODEL_NAMES}
test_data_by_model = {name: build_dataset(test_df, name, cache_name=f"test_{name}.cache") for name in MODEL_NAMES}

The datasets are created once per preprocessing convention so evaluation uses the same input rule as training. If the cache files or memory footprint are too large for a particular Kaggle runtime, set `cache_datasets` to `False` in the configuration and rerun from the top.

## Train the Baseline CNN

The baseline is intentionally ordinary: four convolutional blocks, batch normalization, pooling, dropout, and a softmax head. It gives the transfer models a meaningful from-scratch reference.

In [ ]:
baseline = build_baseline()
baseline_history = fit_model(
    baseline,
    train_data_by_model["baseline_cnn"],
    val_data_by_model["baseline_cnn"],
    "baseline_cnn",
    CONFIG["stage1_epochs"],
    CONFIG["scratch_learning_rate"],
)

## Train Transfer Models

Both transfer models use the same two-stage recipe: train a new head with the ImageNet backbone frozen, then unfreeze only the top backbone layers at a lower learning rate. Validation loss controls checkpointing and early stopping; test data is still untouched.

In [ ]:
mobilenet_model, mobilenet_stage1, mobilenet_stage2 = fit_transfer_two_stage(
    "mobilenetv2",
    train_data_by_model["mobilenetv2"],
    val_data_by_model["mobilenetv2"],
    "mobilenetv2",
    unfrozen_layers=40,
    stage2_epochs=CONFIG["stage2_epochs"],
)

efficientnet_model, efficientnet_stage1, efficientnet_stage2 = fit_transfer_two_stage(
    "efficientnetb0",
    train_data_by_model["efficientnetb0"],
    val_data_by_model["efficientnetb0"],
    "efficientnetb0",
    unfrozen_layers=40,
    stage2_epochs=CONFIG["stage2_epochs"],
)

In [ ]:
def best_validation_accuracy(history):
    values = history.history.get("val_accuracy", [])
    return max(values) if values else -np.inf


validation_scores = {
    "baseline_cnn": best_validation_accuracy(baseline_history),
    "mobilenetv2": max(best_validation_accuracy(mobilenet_stage1), best_validation_accuracy(mobilenet_stage2)),
    "efficientnetb0": max(best_validation_accuracy(efficientnet_stage1), best_validation_accuracy(efficientnet_stage2)),
}
best_transfer_name = max(["mobilenetv2", "efficientnetb0"], key=validation_scores.get)
print("Best transfer family selected using validation accuracy:", best_transfer_name)

The comparison decision is made from validation history only. The held-out test split is reserved for the final model reports below, so it cannot quietly influence architecture selection.

## Evaluation Helpers

Metrics are computed with NumPy from one-hot-free integer labels and predictions. This avoids scikit-learn and makes the macro/weighted averaging formulas visible for interview discussion.

In [ ]:
def predict_dataset(model, dataset):
    labels = []
    probabilities = []
    for images, batch_labels in dataset:
        labels.append(batch_labels.numpy())
        probabilities.append(model(images, training=False).numpy())
    return np.concatenate(labels), np.concatenate(probabilities)


def confusion_matrix_numpy(labels, predictions, number_of_classes):
    matrix = np.zeros((number_of_classes, number_of_classes), dtype=np.int64)
    np.add.at(matrix, (labels, predictions), 1)
    return matrix


def per_class_metrics(labels, predictions, number_of_classes):
    matrix = confusion_matrix_numpy(labels, predictions, number_of_classes)
    true_positive = np.diag(matrix).astype(float)
    support = matrix.sum(axis=1).astype(float)
    predicted = matrix.sum(axis=0).astype(float)
    precision = np.divide(true_positive, predicted, out=np.zeros_like(true_positive), where=predicted != 0)
    recall = np.divide(true_positive, support, out=np.zeros_like(true_positive), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
    table = pd.DataFrame({"class": class_names, "precision": precision, "recall": recall, "f1": f1, "support": support.astype(int)})
    macro_f1 = float(f1.mean())
    weighted_f1 = float(np.average(f1, weights=support))
    accuracy = float(np.mean(labels == predictions))
    return accuracy, macro_f1, weighted_f1, table, matrix


def evaluate_model(model, dataset, model_name):
    labels, probabilities = predict_dataset(model, dataset)
    predictions = probabilities.argmax(axis=1)
    accuracy, macro_f1, weighted_f1, table, matrix = per_class_metrics(labels, predictions, NUM_CLASSES)
    return {
        "model": model_name,
        "test_accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "labels": labels,
        "predictions": predictions,
        "probabilities": probabilities,
        "per_class": table,
        "confusion_matrix": matrix,
    }

## Test Comparison

Every trained comparison model is evaluated once on the held-out test split. The saved comparison table contains real values only after the notebook is run; this source notebook intentionally has no precomputed outputs.

In [ ]:
comparison_models = {
    "baseline_cnn": (baseline, test_data_by_model["baseline_cnn"]),
    "mobilenetv2": (mobilenet_model, test_data_by_model["mobilenetv2"]),
    "efficientnetb0": (efficientnet_model, test_data_by_model["efficientnetb0"]),
}
comparison_results = {name: evaluate_model(model, data, name) for name, (model, data) in comparison_models.items()}
comparison_table = pd.DataFrame([{key: result[key] for key in ["model", "test_accuracy", "macro_f1", "weighted_f1"]} for result in comparison_results.values()])
comparison_table.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
display(comparison_table)

comparison_table.set_index("model")[["test_accuracy", "macro_f1", "weighted_f1"]].plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
plt.ylabel("Score")
plt.title("Held-out test comparison")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=160)
plt.show()

The comparison table reports test accuracy, macro F1, and weighted F1 for all three model families. Macro F1 gives each class equal weight, while weighted F1 reflects the class supports in the test split.

## Ablation A: Extra Contrast Normalization

This ablation asks one narrow preprocessing question: does an additional TensorFlow contrast adjustment help the selected transfer family under the same split and training recipe? It is deliberately reported as a difference, not as a causal claim.

In [ ]:
contrast_train = build_dataset(train_df, best_transfer_name, training=True, extra_contrast=True, cache_name="ablation_contrast_train.cache")
contrast_val = build_dataset(val_df, best_transfer_name, extra_contrast=True, cache_name="ablation_contrast_val.cache")
contrast_test = build_dataset(test_df, best_transfer_name, extra_contrast=True, cache_name="ablation_contrast_test.cache")
contrast_model, contrast_stage1, contrast_stage2 = fit_transfer_two_stage(
    best_transfer_name,
    contrast_train,
    contrast_val,
    "ablation_contrast",
    unfrozen_layers=40,
    stage1_epochs=CONFIG["ablation_stage1_epochs"],
    stage2_epochs=CONFIG["ablation_stage2_epochs"],
)
contrast_result = evaluate_model(contrast_model, contrast_test, "ablation_contrast")
ablation_a = pd.DataFrame([
    {"model": best_transfer_name, "preprocessing": "standard", "test_accuracy": comparison_results[best_transfer_name]["test_accuracy"], "macro_f1": comparison_results[best_transfer_name]["macro_f1"]},
    {"model": best_transfer_name, "preprocessing": "extra_contrast", "test_accuracy": contrast_result["test_accuracy"], "macro_f1": contrast_result["macro_f1"]},
])
ablation_a["accuracy_difference_vs_standard"] = ablation_a["test_accuracy"] - ablation_a.iloc[0]["test_accuracy"]
ablation_a.to_csv(OUTPUT_DIR / "ablation_contrast.csv", index=False)
display(ablation_a)

The ablation reuses the selected family but changes only the extra contrast operation. Its result should be interpreted together with training time and variability; one run cannot prove that contrast normalization is universally beneficial.

## Ablation B: Frozen EfficientNet Embeddings

The second ablation freezes EfficientNetB0, extracts its global-average-pooled features, and trains one Dense softmax layer. This is a simple Keras equivalent of a multinomial linear classifier on fixed embeddings.

In [ ]:
embedding_backbone = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
embedding_backbone.trainable = False
embedding_extractor = tf.keras.Sequential([embedding_backbone, tf.keras.layers.GlobalAveragePooling2D()], name="efficientnet_embeddings")


def extract_embeddings(dataset):
    features, labels = [], []
    for images, batch_labels in dataset:
        features.append(embedding_extractor(images, training=False).numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(features), np.concatenate(labels)


efficient_train_features, efficient_train_labels = extract_embeddings(build_dataset(train_df, "efficientnetb0", cache_name="embedding_train.cache"))
efficient_val_features, efficient_val_labels = extract_embeddings(build_dataset(val_df, "efficientnetb0", cache_name="embedding_val.cache"))
efficient_test_features, efficient_test_labels = extract_embeddings(build_dataset(test_df, "efficientnetb0", cache_name="embedding_test.cache"))

embedding_classifier = tf.keras.Sequential([
    tf.keras.Input(shape=(efficient_train_features.shape[1],)),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32"),
], name="embedding_softmax")
embedding_classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
embedding_classifier.fit(
    efficient_train_features,
    efficient_train_labels,
    validation_data=(efficient_val_features, efficient_val_labels),
    epochs=CONFIG["stage1_epochs"],
    class_weight=class_weight,
    callbacks=make_callbacks("embedding_softmax"),
    verbose=1,
)
embedding_probabilities = embedding_classifier.predict(efficient_test_features, verbose=0)
embedding_accuracy, embedding_macro_f1, embedding_weighted_f1, _, _ = per_class_metrics(efficient_test_labels, embedding_probabilities.argmax(axis=1), NUM_CLASSES)
ablation_b = pd.DataFrame([{"model": "efficientnet_frozen_embeddings", "test_accuracy": embedding_accuracy, "macro_f1": embedding_macro_f1, "weighted_f1": embedding_weighted_f1}])
ablation_b.to_csv(OUTPUT_DIR / "ablation_embeddings.csv", index=False)
display(ablation_b)

This experiment separates representation quality from end-to-end adaptation. A frozen embedding classifier is fast and interpretable, but it cannot adjust convolutional features to the dataset.

## Tune the Best Transfer Family

Tuning is restricted to a stratified 20% subset of the training split. The search varies learning rate, dropout, and the number of trainable backbone layers; validation data is used for ranking, and the test set remains unopened until final evaluation.

In [ ]:
tuning_train_df = train_df.groupby("label", group_keys=False).sample(frac=CONFIG["tuning_subset_fraction"], random_state=SEED).reset_index(drop=True)
tuning_train = build_dataset(tuning_train_df, best_transfer_name, training=True, cache_name="tuning_train.cache")
tuning_val = val_data_by_model[best_transfer_name]


def sample_tuning_config(rng):
    return {
        "learning_rate": float(rng.choice(CONFIG["tuning_learning_rates"])),
        "dropout": float(rng.choice(CONFIG["dropout_values"])),
        "unfrozen_layers": int(rng.choice(CONFIG["unfrozen_layer_values"])),
    }


def build_tuning_model(config):
    return compile_model(build_transfer_model(best_transfer_name, config["dropout"], config["unfrozen_layers"]), config["learning_rate"])


tuning_rows = []
try:
    import keras_tuner as kt

    def hypermodel_builder(hp):
        config = {
            "learning_rate": hp.Choice("learning_rate", CONFIG["tuning_learning_rates"]),
            "dropout": hp.Choice("dropout", CONFIG["dropout_values"]),
            "unfrozen_layers": hp.Choice("unfrozen_layers", CONFIG["unfrozen_layer_values"]),
        }
        return build_tuning_model(config)

    tuner = kt.RandomSearch(hypermodel_builder, objective="val_accuracy", max_trials=CONFIG["tuning_trials"], directory="tuner_runs", project_name="plant_disease", overwrite=True, seed=SEED)
    tuner.search(tuning_train, validation_data=tuning_val, epochs=CONFIG["tuning_epochs"], verbose=1)
    for trial in tuner.oracle.get_best_trials(num_trials=CONFIG["tuning_trials"]):
        tuning_rows.append({"trial_id": trial.trial_id, "score": trial.score, **trial.hyperparameters.values})
    best_config = tuner.get_best_hyperparameters(1)[0].values
except ImportError:
    rng = np.random.default_rng(SEED)
    for trial_id in range(CONFIG["tuning_trials"]):
        config = sample_tuning_config(rng)
        candidate = build_tuning_model(config)
        history = candidate.fit(tuning_train, validation_data=tuning_val, epochs=CONFIG["tuning_epochs"], class_weight=class_weight, verbose=0)
        score = max(history.history["val_accuracy"])
        tuning_rows.append({"trial_id": trial_id, "score": score, **config})
    best_config = max(tuning_rows, key=lambda row: row["score"])

tuning_trials = pd.DataFrame(tuning_rows).sort_values("score", ascending=False)
tuning_trials.to_csv(OUTPUT_DIR / "tuning_trials.csv", index=False)
display(tuning_trials)
print("Best tuning configuration:", best_config)

In [ ]:
final_train = train_data_by_model[best_transfer_name]
final_val = val_data_by_model[best_transfer_name]
final_test = test_data_by_model[best_transfer_name]
final_model = build_transfer_model(best_transfer_name, best_config["dropout"], best_config["unfrozen_layers"])
final_history = fit_model(final_model, final_train, final_val, "best_model", CONFIG["stage2_epochs"], best_config["learning_rate"])
final_result = evaluate_model(final_model, final_test, "best_model")
final_model.save(OUTPUT_DIR / "best_model.keras")

The tuned configuration is retrained on the full training split before the final test pass. This is the model whose artifact and detailed diagnostics should be used in the README after you run the notebook.

## Detailed Final Evaluation

The following diagnostics use the final model's one held-out test evaluation. They include the per-class table, confusion matrix, most-confused pairs, and highest-confidence mistakes requested for the project report.

In [ ]:
final_result["per_class"].to_csv(OUTPUT_DIR / "per_class_metrics.csv", index=False)
pd.DataFrame(final_result["confusion_matrix"], index=class_names, columns=class_names).to_csv(OUTPUT_DIR / "confusion_matrix.csv")

plt.figure(figsize=(14, 12))
plt.imshow(final_result["confusion_matrix"], cmap="Blues")
plt.colorbar(label="Images")
plt.xticks(range(NUM_CLASSES), class_names, rotation=90, fontsize=6)
plt.yticks(range(NUM_CLASSES), class_names, fontsize=6)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Final model confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180)
plt.show()

pair_rows = []
matrix = final_result["confusion_matrix"].copy()
np.fill_diagonal(matrix, 0)
for flat_index in np.argsort(matrix.ravel())[::-1][:10]:
    true_index, predicted_index = np.unravel_index(flat_index, matrix.shape)
    if matrix[true_index, predicted_index] > 0:
        pair_rows.append({"true_class": class_names[true_index], "predicted_class": class_names[predicted_index], "count": int(matrix[true_index, predicted_index])})
top_confused_pairs = pd.DataFrame(pair_rows)
top_confused_pairs.to_csv(OUTPUT_DIR / "top_confused_pairs.csv", index=False)
display(top_confused_pairs)

wrong_indices = np.where(final_result["labels"] != final_result["predictions"])[0]
confidence_order = wrong_indices[np.argsort(final_result["probabilities"][wrong_indices].max(axis=1))[::-1]][:10]
wrong_rows = pd.DataFrame({"test_index": confidence_order, "true_class": [class_names[final_result["labels"][i]] for i in confidence_order], "predicted_class": [class_names[final_result["predictions"][i]] for i in confidence_order], "confidence": [float(final_result["probabilities"][i].max()) for i in confidence_order]})
wrong_rows.to_csv(OUTPUT_DIR / "high_confidence_wrong_predictions.csv", index=False)
display(wrong_rows)

The detailed tables are saved as CSV files so they can be pasted into the experiment report. Confusion-pair counts describe what the model confuses; they do not by themselves explain why the confusion occurs.

## Grad-CAM

Grad-CAM highlights the spatial regions that most affect a chosen prediction. We show six correct and six incorrect test examples; any written error interpretation should be based on these visualizations and the corresponding counts, otherwise the report should say `not determined`.

In [ ]:
def raw_image_for_record(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    return tf.image.resize(image, IMAGE_SIZE)


def model_input_for_record(path, model_name):
    image = raw_image_for_record(path)
    return preprocess_images(image, model_name)[None, ...]


def last_four_dimensional_layer(model):
    for layer in reversed(model.layers):
        if hasattr(layer, "output_shape") and len(layer.output_shape) == 4:
            return layer
    return None


def make_gradcam_heatmap(model, image_batch, class_index):
    conv_layer = last_four_dimensional_layer(model)
    if conv_layer is None:
        raise ValueError("Could not find a convolutional feature layer for Grad-CAM.")
    grad_model = tf.keras.Model(model.inputs, [conv_layer.output, model.output])
    with tf.GradientTape() as tape:
        convolution_output, predictions = grad_model(image_batch)
        score = predictions[:, class_index]
    gradients = tape.gradient(score, convolution_output)
    weights = tf.reduce_mean(gradients, axis=(1, 2))
    heatmap = tf.reduce_sum(convolution_output * weights[:, None, None, :], axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap, axis=(1, 2), keepdims=True) + 1e-8
    return heatmap[0].numpy()


def overlay_heatmap(image, heatmap):
    heatmap = np.uint8(255 * heatmap)
    colored = plt.get_cmap("jet")(heatmap)[..., :3]
    colored = tf.image.resize(colored, IMAGE_SIZE).numpy()
    base = np.clip(image.numpy() / 255.0, 0, 1)
    return np.clip(0.55 * base + 0.45 * colored, 0, 1)


def save_gradcam_grid(indices, title, filename):
    figure, axes = plt.subplots(2, 3, figsize=(13, 8))
    for axis, index in zip(axes.flat, indices):
        record = test_df.iloc[int(index)]
        image = raw_image_for_record(record["path"])
        input_batch = model_input_for_record(record["path"], best_transfer_name)
        predicted_class = int(final_result["predictions"][index])
        heatmap = make_gradcam_heatmap(final_model, input_batch, predicted_class)
        axis.imshow(overlay_heatmap(image, heatmap))
        axis.set_title(f"true: {class_names[record['label_id']]}\npred: {class_names[predicted_class]}", fontsize=8)
        axis.axis("off")
    figure.suptitle(title)
    plt.tight_layout()
    figure.savefig(OUTPUT_DIR / filename, dpi=160)
    plt.show()


correct_indices = np.where(final_result["labels"] == final_result["predictions"])[0][:6]
incorrect_indices = np.where(final_result["labels"] != final_result["predictions"])[0][:6]
save_gradcam_grid(correct_indices, "Grad-CAM: correct predictions", "gradcam_correct.png")
save_gradcam_grid(incorrect_indices, "Grad-CAM: incorrect predictions", "gradcam_incorrect.png")

The final cell writes the requested model, metric files, plots, tuning table, and Grad-CAM figures into `results/`. Run the notebook top to bottom on Kaggle, then copy the generated values and images into the README placeholders.